In [ ]:
"""
File này sẽ dùng sapbert để kiểm tra tác vụ retrieval trên 12 entities khác nhau.
Lý do chọn sapbert là hoàn thành tốt retrieval task trên dữ liệu tiếng anh.
"""

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

def embed_texts(texts):
    """
    Convert a list of medical texts into normalized SapBERT embeddings.
    """
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        # SapBERT commonly uses the [CLS] token embedding
        embeddings = outputs.last_hidden_state[:, 0, :]

        # Normalize so dot product = cosine similarity
        embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings

# Pretend these come from your VinAI Vietnamese -> English translation step
queries = [
    "high blood pressure",
    "type 2 diabetes",
    "shortness of breath",
    "amoxicillin allergy",
    "acetaminophen"
]

# Mini fake RxNorm / ICD-10-style candidate database
candidates = [
    "hypertension",
    "essential primary hypertension",
    "diabetes mellitus type 2",
    "non-insulin-dependent diabetes mellitus",
    "dyspnea",
    "difficulty breathing",
    "allergy to amoxicillin",
    "penicillin allergy",
    "acetaminophen",
    "paracetamol",
    "ibuprofen",
    "acute myocardial infarction"
]

query_embeddings = embed_texts(queries)
candidate_embeddings = embed_texts(candidates)

# Cosine similarity: each query against every candidate
similarity_matrix = query_embeddings @ candidate_embeddings.T

TOP_K = 3

for i, query in enumerate(queries):
    scores, indices = torch.topk(similarity_matrix[i], k=TOP_K)

    print(f"\nQuery: {query}")
    for rank, (score, idx) in enumerate(zip(scores, indices), start=1):
        print(f"  {rank}. {candidates[idx]:45} score = {score.item():.4f}")

Query: high blood pressure
  1. hypertension                                  score = 0.8946
  2. essential primary hypertension                score = 0.6731
  3. diabetes mellitus type 2                      score = 0.2933

Query: type 2 diabetes
  1. diabetes mellitus type 2                      score = 0.9795
  2. non-insulin-dependent diabetes mellitus       score = 0.8104
  3. essential primary hypertension                score = 0.3782

Query: shortness of breath
  1. difficulty breathing                          score = 0.7870
  2. dyspnea                                       score = 0.7474
  3. allergy to amoxicillin                        score = 0.2789

Query: amoxicillin allergy
  1. allergy to amoxicillin                        score = 0.9802
  2. penicillin allergy                            score = 0.5852
  3. acute myocardial infarction                   score = 0.2367

Query: acetaminophen
  1. acetaminophen                                 score = 1.0000
  2. paracetamol                                   score = 0.9420
  3. ibuprofen                                     score = 0.3565

### Đánh giá: 
Chạy khá tốt